In [22]:
import os
from dotenv import load_dotenv

load_dotenv()

google_api_key=os.getenv("GOOGLE_API_KEY")



In [31]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from langchain_groq import ChatGroq

# llm = ChatGoogleGenerativeAI(
#     model="gemini-1.5-flash",  # NOTE: correct model name
#     google_api_key="google_api_key"
# )
llm=ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY")
)

agent=create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    # middleware=[
    #     SummarizationMiddleware(
    #         model=llm,
    #         trigger=("messages",10),
    #         keep=("messages",4)
    #     )
    # ]

)

In [27]:
config={"configurable":{"thread_id":"test-1"}}

In [32]:
questions=[
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",

]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)

    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='7de41add-e9f9-487d-b450-b277d9152799'), AIMessage(content='2 + 2 = 4.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 42, 'total_tokens': 51, 'completion_time': 0.012288076, 'completion_tokens_details': None, 'prompt_time': 0.001190881, 'prompt_tokens_details': None, 'queue_time': 0.055476678, 'total_time': 0.013478957}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e5ae6-e828-7153-adf4-3789921ff62c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 42, 'output_tokens': 9, 'total_tokens': 51})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='7de41add-e9f9-487d-b450-b277d9152799'), AIMess

In [38]:
from langchain_core.callbacks import LLMManagerMixin
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import create_react_agent

@tool
def search_hotels(city:str)->str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent=create_react_agent(
    model=llm,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config={"configurable":{"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars // 4
    




C:\Users\harsh\AppData\Local\Temp\ipykernel_4316\1668949925.py:17: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent=create_react_agent(


TypeError: create_react_agent() got unexpected keyword arguments: {'middleware': [<langchain.agents.middleware.summarization.SummarizationMiddleware object at 0x000001E28C162010>]}

In [39]:
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities :
    response = agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: {tokens} tokens, {len(response['messages'])}messages")
    print(f"{(response['messages'])}")

Paris: 186 tokens, 8messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='bf7fd0e9-7bc1-475e-933e-56e4b7a5ed38'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '2s7767age', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 226, 'total_tokens': 241, 'completion_time': 0.039054875, 'completion_tokens_details': None, 'prompt_time': 0.022149737, 'prompt_tokens_details': None, 'queue_time': 0.056363552, 'total_time': 0.061204612}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e5b18-0fa8-7441-8924-4bdec6a6f351-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': '2s7767age', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata